<a href="https://colab.research.google.com/github/legallyananyar-dev/digital-twin-challenge-/blob/main/WarfarinTwin_02_digital_twin_simulation_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WarfarinTwin, Notebook 2: The Digital Twin (INR forecasting, alerts, what-if)

## 1. Setup

In [ ]:
!pip -q install xlrd

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib
from scipy.stats import norm
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng_global = np.random.default_rng(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True); os.makedirs("models", exist_ok=True)

### Load the IWPC file and your Notebook 1 model

In [ ]:
# ---------- locate files ----------
def find_file(pattern, label):
    hits = glob.glob(pattern) + glob.glob("/content/" + pattern) + glob.glob("/content/**/" + pattern, recursive=True)
    if hits:
        return hits[0]
    from google.colab import files
    print(f"Upload {label}")
    up = files.upload()
    return list(up.keys())[0]

IWPC_PATH = find_file("PS206767-553247439.xls", "the IWPC .xls file")
raw = pd.read_excel(IWPC_PATH, sheet_name="Subject Data", engine="xlrd")

NUM = ["age_decade", "height_cm", "weight_kg", "male"]
BIN = ["inducer", "amiodarone"]
CAT = ["race", "cyp2c9", "vkorc1"]
FEATS = NUM + BIN + CAT

def clean_iwpc(raw):
    d = pd.DataFrame(index=raw.index)
    d["age_decade"] = raw["Age"].astype(str).str.extract(r"(\d+)")[0].astype(float) // 10
    d["height_cm"] = pd.to_numeric(raw["Height (cm)"], errors="coerce")
    d["weight_kg"] = pd.to_numeric(raw["Weight (kg)"], errors="coerce")
    d["male"] = raw["Gender"].map({"male": 1.0, "female": 0.0})
    race = raw["Race (OMB)"].astype(str)
    d["race"] = np.select([race.str.contains("Asian", case=False), race.str.contains("Black", case=False),
                           race.str.contains("White", case=False)], ["Asian", "Black", "White"], default="Unknown")
    d["inducer"] = raw[["Carbamazepine (Tegretol)", "Phenytoin (Dilantin)", "Rifampin or Rifampicin"]].fillna(0).max(axis=1)
    d["amiodarone"] = pd.to_numeric(raw["Amiodarone (Cordarone)"], errors="coerce").fillna(0)
    cyp_ok = ["*1/*1", "*1/*2", "*1/*3", "*2/*2", "*2/*3", "*3/*3"]
    d["cyp2c9"] = raw["CYP2C9 consensus"].where(raw["CYP2C9 consensus"].isin(cyp_ok), "Unknown")
    d["vkorc1"] = raw["VKORC1 -1639 consensus"].where(raw["VKORC1 -1639 consensus"].isin(["A/A", "A/G", "G/G"]), "Unknown")
    d["dose_week"] = pd.to_numeric(raw["Therapeutic Dose of Warfarin"], errors="coerce")
    d["stable"] = raw["Subject Reached Stable Dose of Warfarin"]
    return d

iwpc = clean_iwpc(raw)
iwpc = iwpc[(iwpc["stable"] == 1) & iwpc["dose_week"].notna()].drop(columns="stable").reset_index(drop=True)
print("IWPC patients:", len(iwpc))

In [ ]:
# ---------- starting-dose model from Notebook 1 (or a quick fallback) ----------
def apply_imputer(imp, df):
    out = df.copy()
    out[NUM] = imp.transform(df[NUM])
    out["male"] = out["male"].clip(0, 1)
    return out

MODEL_PATH = (glob.glob("models/starting_dose_model.joblib") + glob.glob("/content/**/starting_dose_model.joblib", recursive=True) + [None])[0]
if MODEL_PATH is None:
    print("Notebook 1 model not found -> training a quick Ridge fallback (upload models/starting_dose_model.joblib to use the full model).")
    imp0 = IterativeImputer(max_iter=10, random_state=RANDOM_STATE).fit(iwpc[NUM])
    tmp = apply_imputer(imp0, iwpc)
    enc = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough", verbose_feature_names_out=False)
    m0 = Pipeline([("enc", enc), ("reg", RidgeCV(alphas=np.logspace(-2, 3, 20)))]).fit(tmp[FEATS], np.sqrt(tmp["dose_week"]))
    bundle = {"imputer": imp0, "model": m0, "features": FEATS}
else:
    bundle = joblib.load(MODEL_PATH)
    print("Loaded starting-dose model:", MODEL_PATH)

def predict_dose_week(df):
    """Notebook-1 model: stable weekly dose (mg) for a dataframe with the FEATS columns."""
    row = apply_imputer(bundle["imputer"], df[FEATS])
    return np.clip(bundle["model"].predict(row[bundle["features"]]), 0, None) ** 2

## 2. The virtual-patient simulator

In [ ]:
# ---------- simulator constants ----------
N_DAYS = 12            # INR measured on mornings of days 0..11
TARGET_INR = 2.5
IMAX, GAMMA = 12.0, 2.5
F_TARGET = (TARGET_INR - 1.0) / IMAX
EC50 = ((1 - F_TARGET) / F_TARGET) ** (1 / GAMMA)     # makes INR = TARGET when normalized exposure = 1
K0 = 0.55              # 1/day, S-warfarin-like elimination (half-life ~1.3 days) for a normal metabolizer
MTT0 = 2.5             # days, mean transit time of the response delay chain
N_TRANSIT = 3
# ASSUMED relative elimination rates by CYP2C9 genotype (illustrative; verify / run sensitivity analysis)
CL_MULT = {"*1/*1": 1.00, "*1/*2": 0.85, "*1/*3": 0.65, "*2/*2": 0.70, "*2/*3": 0.50, "*3/*3": 0.30, "Unknown": 0.90}
SIG_EPS = 0.06         # proportional INR measurement noise
SIG_TOTAL = 0.12       # the twin's own (inflated) belief about forecast noise
VAR_SCALE = 0.7        # ASSUMPTION: share of IWPC dose-residual spread treated as real biology (rest = record noise)

def inr_from_exposure(c):
    cg = np.power(np.clip(c, 0, None), GAMMA)
    return 1.0 + IMAX * cg / (cg + EC50 ** GAMMA)

In [ ]:
def simulate(p, policy, n_days=N_DAYS):
    """Hourly simulation of N patients.
    p: dict of arrays: dstar (true stable dose, mg/day), k, mtt, u (N,n_days), ev_day, ev_mult, eps (N,n_days)
    policy(day, inr_obs_history, dose_history) -> dose (mg) given the evening of `day`."""
    N = len(p["dstar"]); dt = 1.0 / 24
    A = np.zeros(N); T = np.zeros((N_TRANSIT, N))
    ktr = N_TRANSIT / p["mtt"]; ek = np.exp(-p["k"] * dt); kscale = p["k"] / p["dstar"]
    inr_true = np.zeros((N, n_days)); inr_obs = np.zeros((N, n_days)); dose = np.zeros((N, n_days - 1))
    for day in range(n_days):
        mult = np.exp(p["u"][:, day]) * np.where(day >= p["ev_day"], p["ev_mult"], 1.0)
        for hour in range(24):
            if hour == 0:                                   # 08:00 morning blood draw
                inr_true[:, day] = inr_from_exposure(T[-1] * mult)
                inr_obs[:, day] = inr_true[:, day] * np.exp(p["eps"][:, day])
            if hour == 10 and day < n_days - 1:             # 18:00 evening dose
                dose[:, day] = policy(day, inr_obs[:, :day + 1], dose[:, :day])
                A = A + dose[:, day]
            A = A * ek
            C = A * kscale                                  # normalized exposure (1 = at stable dose)
            T[0] += ktr * dt * (C - T[0])
            for j in range(1, N_TRANSIT):
                T[j] += ktr * dt * (T[j - 1] - T[j])
    return {"inr_true": inr_true, "inr_obs": inr_obs, "dose": dose}

class OpenLoop:
    """Policy that replays a fixed dose schedule (used for the twin and for what-if tests)."""
    def __init__(self, sched): self.sched = sched
    def __call__(self, day, inr_hist, dose_hist): return self.sched[:, day]

In [ ]:
class TitrationPolicy:
    """Simple inpatient-style titration. ASSUMPTION: a plausible protocol, not a published nomogram.
    Day 0-2: fixed starting dose. From day 3: raise on odd days if INR<1.8; trim if INR>=2.8;
    half dose tonight if 3.2<=INR<3.5; hold tonight if INR>=3.5 (and cut the maintenance dose)."""
    def __init__(self, start, explore_sd, rng):
        self.base = np.asarray(start, float).copy(); self.sd = np.asarray(explore_sd, float); self.rng = rng
    def __call__(self, day, inr_hist, dose_hist):
        inr = inr_hist[:, -1]; tonight = np.ones_like(inr)
        if day >= 3:
            up = (inr < 1.8) & (day % 2 == 1)
            fac = np.ones_like(inr)
            fac = np.where(up & (inr < 1.5), 1.20, fac)
            fac = np.where(up & (inr >= 1.5), 1.10, fac)
            fac = np.where((inr >= 2.8) & (inr < 3.2), 0.90, fac)
            fac = np.where((inr >= 3.2) & (inr < 3.5), 0.80, fac)
            fac = np.where(inr >= 3.5, 0.72, fac)
            tonight = np.where((inr >= 3.2) & (inr < 3.5), 0.5, tonight)
            tonight = np.where(inr >= 3.5, 0.0, tonight)
            self.base = np.clip(self.base * fac, 0.5, 15.0)
        dose = self.base * tonight * np.exp(self.rng.normal(0, 1, len(self.base)) * self.sd)
        return np.clip(np.round(dose * 2) / 2, 0.0, 15.0)

In [ ]:
# ---------- virtual cohort ----------
POOL = iwpc[(iwpc["cyp2c9"] != "Unknown") & (iwpc["vkorc1"] != "Unknown")].reset_index(drop=True)
pool_pred = predict_dose_week(POOL)
RESID = np.clip(np.log(POOL["dose_week"].values / np.clip(pool_pred, 1.0, None)), -1.0, 1.0)
print("Pool with known genotypes:", len(POOL), "| residual SD (log scale): %.3f" % RESID.std())

def sample_cohort(n, rng, ancestry="all", age_mean=72, age_sd=6):
    pool = POOL if ancestry == "all" else POOL[POOL["race"] == ancestry]
    c = pool.sample(n, replace=True, random_state=int(rng.integers(1e9))).reset_index(drop=True)
    c = c.drop(columns=["dose_week"])
    c["age"] = np.clip(rng.normal(age_mean, age_sd, n), 65, 90)
    c["age_decade"] = np.floor(c["age"] / 10)
    c[NUM] = bundle["imputer"].transform(c[NUM]); c["male"] = c["male"].clip(0, 1)
    hidden = c.copy(); hidden["cyp2c9"] = "Unknown"; hidden["vkorc1"] = "Unknown"
    c["dpred_true"] = predict_dose_week(c) / 7.0           # mg/day, using the TRUE genotype
    c["dpred_hidden"] = predict_dose_week(hidden) / 7.0    # mg/day, genotype not available
    eta = VAR_SCALE * rng.choice(RESID, n)                              # unexplained between-patient variability (data-driven)
    c["dstar"] = c["dpred_true"] * np.exp(eta)              # patient's TRUE stable dose (hidden from the models)
    c["k"] = K0 * c["cyp2c9"].map(CL_MULT) * np.exp(rng.normal(0, 0.20, n))
    c["mtt"] = MTT0 * np.exp(rng.normal(0, 0.20, n))
    u = np.zeros((n, N_DAYS))                               # day-to-day sensitivity wobble (diet, illness)
    for t in range(1, N_DAYS): u[:, t] = 0.6 * u[:, t - 1] + rng.normal(0, 0.05, n)
    c["ev_day"] = np.where(rng.random(n) < 0.08, rng.integers(3, 8, n), 99)   # 8% start an interacting drug
    c["ev_mult"] = 1.25
    eps = rng.normal(0, SIG_EPS, (n, N_DAYS))
    return c, u, eps

def params_of(c, u, eps):
    return {"dstar": c["dstar"].values, "k": c["k"].values, "mtt": c["mtt"].values, "u": u,
            "ev_day": c["ev_day"].values, "ev_mult": c["ev_mult"].values, "eps": eps}

In [ ]:
# ---------- outcome helpers ----------
def rosendaal_ttr(inr, lo=2.0, hi=3.0, first=4, last=11, step=0.05):
    """Fraction of time in [lo, hi] between day `first` and `last`, linear interpolation between daily INRs."""
    n = inr.shape[0]; tg = np.arange(first, last, step); j = np.floor(tg).astype(int); fr = (tg - j)[None, :]
    v = inr[:, j] + (inr[:, j + 1] - inr[:, j]) * fr
    return ((v >= lo) & (v <= hi)).mean(axis=1)

def start_dose_rounded(x): return np.clip(np.round(np.asarray(x) * 2) / 2, 1.0, 10.0)

### Sanity check the simulator

In [ ]:
# Sanity check: with NO noise, what does the simulated INR do at (a) each patient's true stable dose and (b) a flat 5 mg/day?
c, u, eps = sample_cohort(500, np.random.default_rng(1)); p = params_of(c, u, eps)
p0 = dict(p); p0["u"] = np.zeros_like(u); p0["eps"] = np.zeros_like(eps); p0["ev_day"] = np.full(500, 99)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for a_, (label, sched) in zip(ax, [("dose = patient's TRUE stable dose", np.tile(c["dstar"].values[:, None], (1, N_DAYS - 1))),
                                   ("flat 5 mg/day", np.full((500, N_DAYS - 1), 5.0))]):
    o = simulate(p0, OpenLoop(sched))["inr_true"]
    lo, med, hi = np.percentile(o, [10, 50, 90], axis=0)
    a_.fill_between(range(N_DAYS), lo, hi, alpha=0.25); a_.plot(med, "o-"); a_.axhspan(2, 3, color="green", alpha=0.08)
    a_.axhline(4, color="red", ls=":"); a_.set_title(label); a_.set_xlabel("day")
    print(f"{label}: median INR on day 11 = {med[-1]:.2f} | 10-90% range {lo[-1]:.2f}-{hi[-1]:.2f}")
ax[0].set_ylabel("INR (median and 10-90% band)")
plt.tight_layout(); plt.savefig("outputs/simulator_sanity.png", dpi=150); plt.show()

## 3. In-silico mini-trial (GIFT-style comparison)

In [ ]:
# ---------- in-silico mini-trial: SAME virtual patients under two strategies ----------
N_TRIAL = 1500
ct, ut, et = sample_cohort(N_TRIAL, np.random.default_rng(101))
pt = params_of(ct, ut, et)
arms = {"Flat 5 mg/day start (reference)": np.full(N_TRIAL, 5.0),
        "Clinical-factors start (no genotype)": start_dose_rounded(ct["dpred_hidden"]),
        "Genotype-guided start": start_dose_rounded(ct["dpred_true"])}
trial_out = {}
for name, start in arms.items():
    trial_out[name] = simulate(pt, TitrationPolicy(start, np.zeros(N_TRIAL), np.random.default_rng(0)))

rows = []
for name, o in trial_out.items():
    obs = o["inr_obs"]
    rows.append({"arm": name,
                 "INR>=4 (any, days 1-11)": (obs[:, 1:] >= 4).any(axis=1).mean(),
                 "INR>=5 (any)": (obs[:, 1:] >= 5).any(axis=1).mean(),
                 "TTR 2-3 (days 4-11)": rosendaal_ttr(obs).mean(),
                 "mean |INR-2.5| days 4-11": np.abs(obs[:, 4:] - 2.5).mean()})
trial_tbl = pd.DataFrame(rows).set_index("arm").round(3)
trial_tbl.to_csv("outputs/insilico_trial_results.csv")
print(trial_tbl)
a, b = [(trial_out[n]["inr_obs"][:, 1:] >= 4).any(axis=1) for n in ["Clinical-factors start (no genotype)", "Genotype-guided start"]]
print("Relative reduction in INR>=4, genotype vs clinical-factors start: %.1f%%" % (100 * (a.mean() - b.mean()) / a.mean()))
rb = np.random.default_rng(5); diffs = []
for _ in range(1000):
    i = rb.integers(0, N_TRIAL, N_TRIAL); diffs.append(a[i].mean() - b[i].mean())
print("Absolute risk difference: %.3f  (95%% CI %.3f to %.3f)" % (a.mean() - b.mean(), *np.percentile(diffs, [2.5, 97.5])))

# sensitivity of that result to the biggest assumption (how much dose variability is 'real biology')
sens = []
for vs in [0.5, 0.7, 1.0]:
    VAR_SCALE_old = VAR_SCALE; globals()["VAR_SCALE"] = vs
    cs, us, es = sample_cohort(1000, np.random.default_rng(7)); ps = params_of(cs, us, es); r = {}
    for nm, st in [("clin", start_dose_rounded(cs["dpred_hidden"])), ("geno", start_dose_rounded(cs["dpred_true"]))]:
        r[nm] = (simulate(ps, TitrationPolicy(st, np.zeros(1000), np.random.default_rng(0)))["inr_obs"][:, 1:] >= 4).any(axis=1).mean()
    sens.append({"VAR_SCALE": vs, "INR>=4 clinical": r["clin"], "INR>=4 genotype": r["geno"],
                 "relative reduction": (r["clin"] - r["geno"]) / r["clin"]})
    globals()["VAR_SCALE"] = VAR_SCALE_old
print(pd.DataFrame(sens).round(3).to_string(index=False))

## 4. Training cohort and the mechanistic Bayesian twin

In [ ]:
# ================= training cohort (each virtual patient follows ONE dosing strategy) =================
N_TRAIN = 4000
cohort, U, EPS = sample_cohort(N_TRAIN, np.random.default_rng(202))
rg = np.random.default_rng(303)
arm = rg.choice(["clinical", "genotype", "explore"], N_TRAIN, p=[0.40, 0.35, 0.25])
cohort["arm"] = arm
cohort["geno_known"] = np.where(arm == "genotype", 1, np.where(arm == "clinical", 0, rg.integers(0, 2, N_TRAIN)))
known = cohort["geno_known"].values == 1
cohort["dpred_avail"] = np.where(known, cohort["dpred_true"], cohort["dpred_hidden"])        # mg/day the clinician/twin can see
cohort["cyp_avail"] = np.where(known, cohort["cyp2c9"], "Unknown")
cohort["vk_avail"] = np.where(known, cohort["vkorc1"], "Unknown")
start = np.where(arm == "explore", cohort["dpred_hidden"] * np.exp(rg.normal(0, 0.3, N_TRAIN)), cohort["dpred_avail"])
start = start_dose_rounded(start)
explore_sd = np.where(arm == "explore", 0.25, 0.0)   # random dose noise so the data contains varied dose-response information
sim_train = simulate(params_of(cohort, U, EPS), TitrationPolicy(start, explore_sd, np.random.default_rng(404)))
INR_OBS, DOSE, INR_TRUE = sim_train["inr_obs"], sim_train["dose"], sim_train["inr_true"]
print("Simulated", N_TRAIN, "patients; INR>=4 at any point: %.1f%%" % (100 * (INR_OBS[:, 1:] >= 4).any(axis=1).mean()))

In [ ]:
# ================= the mechanistic Bayesian twin =================
Z = np.linspace(-1.2, 1.2, 25)            # grid of "how much more/less warfarin does THIS patient need vs. prediction" (log scale)
z_known = np.log(cohort["dstar"] / cohort["dpred_true"]).std()
z_hidden = np.log(cohort["dstar"] / cohort["dpred_hidden"]).std()
print("Prior SD of log dose-requirement: known genotype %.3f | unknown genotype %.3f" % (z_known, z_hidden))

def static_of(df):
    return {"dpred_avail": df["dpred_avail"].values, "clm_avail": df["cyp_avail"].map(CL_MULT).values,
            "sig": np.where(df["geno_known"].values == 1, z_known, z_hidden)}

def twin_summary(static, inr_obs, sched):
    """For every patient and every origin day d: posterior over the patient's dose requirement given INRs up to day d,
    then forecasts for d+1 and d+2 under the PLANNED doses in `sched`, plus P(INR>=4)."""
    N, G = len(static["dpred_avail"]), len(Z)
    p = {"dstar": (static["dpred_avail"][:, None] * np.exp(Z)[None, :]).ravel(),
         "k": np.repeat(K0 * static["clm_avail"], G), "mtt": np.full(N * G, MTT0),
         "u": np.zeros((N * G, N_DAYS)), "ev_day": np.full(N * G, 99), "ev_mult": np.ones(N * G), "eps": np.zeros((N * G, N_DAYS))}
    pred = simulate(p, OpenLoop(np.repeat(sched, G, axis=0)))["inr_true"].reshape(N, G, N_DAYS)
    sq = (np.log(np.clip(inr_obs, 0.5, None))[:, None, :] - np.log(pred)) ** 2
    sq[:, :, 0] = 0.0
    cum = np.cumsum(sq, axis=2)
    logprior = -0.5 * (Z[None, :] / static["sig"][:, None]) ** 2
    out = {k: np.full((N, N_DAYS), np.nan) for k in ["f1", "f2", "p4_1", "p4_2", "zmean", "zsd"]}
    for d in range(1, N_DAYS - 1):
        ll = -cum[:, :, d] / (2 * SIG_TOTAL ** 2) + logprior
        w = np.exp(ll - ll.max(axis=1, keepdims=True)); w /= w.sum(axis=1, keepdims=True)
        zm = w @ Z; out["zmean"][:, d] = zm; out["zsd"][:, d] = np.sqrt((w * (Z[None, :] - zm[:, None]) ** 2).sum(axis=1))
        for h in (1, 2):
            if d + h < N_DAYS:
                pr = pred[:, :, d + h]
                out[f"f{h}"][:, d] = (w * pr).sum(axis=1)
                out[f"p4_{h}"][:, d] = (w * (1 - norm.cdf((np.log(4.0) - np.log(pr)) / SIG_TOTAL))).sum(axis=1)
    return out

TW = twin_summary(static_of(cohort), INR_OBS, DOSE)
print("Twin built. Example posterior SD of log-dose-requirement at day 2 vs day 8: %.3f -> %.3f" % (np.nanmean(TW["zsd"][:, 2]), np.nanmean(TW["zsd"][:, 8])))

## 5. Forecasting models, patient-level split, and evaluation

In [ ]:
# ================= features =================
RACES, CYPS, VKS = ["Asian", "Black", "White", "Unknown"], list(CL_MULT.keys()), ["A/A", "A/G", "G/G", "Unknown"]

def make_rows(df, inr_obs, dose, tw, origins, h):
    """One row per (patient, origin day d). Information available at the morning of day d plus the PLANNED doses."""
    rows = []
    for d in origins:
        n = len(df); X = pd.DataFrame({"pid": np.arange(n), "day": d})
        X["age"] = df["age"].values; X["height_cm"] = df["height_cm"].values; X["weight_kg"] = df["weight_kg"].values
        X["male"] = df["male"].values; X["geno_known"] = df["geno_known"].values; X["dpred"] = df["dpred_avail"].values
        for r in RACES: X["race_" + r] = (df["race"].values == r).astype(float)
        for c in CYPS: X["cyp_" + c] = (df["cyp_avail"].values == c).astype(float)
        for v in VKS: X["vk_" + v] = (df["vk_avail"].values == v).astype(float)
        X["inr_d"] = inr_obs[:, d]
        X["inr_dm1"] = inr_obs[:, d - 1] if d >= 1 else np.nan
        X["inr_dm2"] = inr_obs[:, d - 2] if d >= 2 else np.nan
        X["dinr"] = X["inr_d"] - X["inr_dm1"]
        X["inr_max"] = inr_obs[:, :d + 1].max(axis=1)
        X["dose_d"] = dose[:, d]                                         # tonight's planned dose
        X["dose_dm1"] = dose[:, d - 1] if d >= 1 else np.nan
        X["dose_dm2"] = dose[:, d - 2] if d >= 2 else np.nan
        X["dose_rel"] = X["dose_d"] / X["dpred"]
        X["cum3"] = dose[:, max(0, d - 2):d + 1].sum(axis=1)
        if h == 2: X["dose_dp1"] = dose[:, d + 1]                      # tomorrow's planned dose
        for k in ["f1", "f2", "p4_1", "p4_2", "zmean", "zsd"]: X["tw_" + k] = tw[k][:, d]
        rows.append(X)
    return pd.concat(rows, ignore_index=True)

BASE_COLS = lambda X: [c for c in X.columns if c not in ["pid", "tw_f1", "tw_f2", "tw_p4_1", "tw_p4_2", "tw_zmean", "tw_zsd"] and not c.startswith("y")]
def twin_cols(h): return ["tw_f%d" % h, "tw_p4_%d" % h, "tw_zmean", "tw_zsd"]

# ================= split by PATIENT (never by row) =================
perm = np.random.default_rng(11).permutation(N_TRAIN)
ids = {"train": perm[:int(.6 * N_TRAIN)], "cal": perm[int(.6 * N_TRAIN):int(.8 * N_TRAIN)], "test": perm[int(.8 * N_TRAIN):]}
DATA = {}
for h, origins in [(1, range(1, 10)), (2, range(1, 10))]:
    X = make_rows(cohort, INR_OBS, DOSE, TW, origins, h)
    X["y"] = INR_OBS[X["pid"].values, X["day"].values + h]
    X["alert"] = (X["y"] >= 4.0).astype(int)
    DATA[h] = {s: X[X["pid"].isin(i)].reset_index(drop=True) for s, i in ids.items()}
print({h: {s: len(v) for s, v in DATA[h].items()} for h in DATA})
print("INR>=4 prevalence (test, h=1 / h=2): %.3f / %.3f" % (DATA[1]["test"]["alert"].mean(), DATA[2]["test"]["alert"].mean()))

In [ ]:
# ================= models =================
def reg(): return HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=RANDOM_STATE)
def clf(): return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=RANDOM_STATE)

def within(y, p, tol=0.5): return float(np.mean(np.abs(y - p) <= tol))
def recall_at_precision(y, s, target=0.5):
    o = np.argsort(-s); ys = y[o]; tp = np.cumsum(ys); prec = tp / (np.arange(len(ys)) + 1)
    ok = np.where(prec >= target)[0]
    return float(tp[ok.max()] / ys.sum()) if len(ok) else 0.0

MODELS, RES, ALERT = {}, [], []
for h in (1, 2):
    D = DATA[h]; tr, ca, te = D["train"], D["cal"], D["test"]
    bc = BASE_COLS(tr); bc = [c for c in bc if c not in ["alert", "y"]]
    hc = bc + twin_cols(h)
    ml = reg().fit(tr[bc], tr["y"])
    hyb = reg().fit(tr[hc], tr["y"] - tr["tw_f%d" % h])                    # hybrid: learn the twin's residual
    MODELS[h] = {"ml": ml, "hyb": hyb, "bc": bc, "hc": hc}
    preds = {"Persistence (INR today)": te["inr_d"].values,
             "Mechanistic twin only": te["tw_f%d" % h].values,
             "ML only (no twin)": ml.predict(te[bc]),
             "Hybrid (twin + ML residual)": te["tw_f%d" % h].values + hyb.predict(te[hc])}
    for name, p in preds.items():
        RES.append({"horizon": f"{h} day", "model": name, "MAE": mean_absolute_error(te["y"], p),
                    "within ±0.5 INR": within(te["y"].values, p), "R2": r2_score(te["y"], p)})
    # conformal (relative) intervals for the hybrid model, calibrated on the CAL patients
    pc = ca["tw_f%d" % h].values + hyb.predict(ca[hc]); rel = np.abs(ca["y"].values - pc) / pc
    q = np.quantile(rel, np.ceil((len(rel) + 1) * 0.9) / len(rel)); MODELS[h]["q90"] = q
    ph = preds["Hybrid (twin + ML residual)"]
    cov = np.mean(np.abs(te["y"].values - ph) <= q * ph)
    RES.append({"horizon": f"{h} day", "model": "Hybrid 90% conformal band", "MAE": np.nan, "within ±0.5 INR": np.nan, "R2": np.nan,
                "coverage": cov, "rel. half-width": q})
    # alerts: P(INR >= 4)
    ac = clf().fit(tr[bc], tr["alert"]); ah = clf().fit(tr[hc], tr["alert"]); MODELS[h]["ac"], MODELS[h]["ah"] = ac, ah
    scores = {"Persistence (INR today)": te["inr_d"].values, "Mechanistic twin only": te["tw_p4_%d" % h].values,
              "ML only (no twin)": ac.predict_proba(te[bc])[:, 1], "Hybrid (twin + ML)": ah.predict_proba(te[hc])[:, 1]}
    for name, s in scores.items():
        ALERT.append({"horizon": f"{h} day", "model": name, "AUROC": roc_auc_score(te["alert"], s),
                      "AUPRC": average_precision_score(te["alert"], s), "recall @ 50% precision": recall_at_precision(te["alert"].values, s)})
res_tbl = pd.DataFrame(RES).round(3); alert_tbl = pd.DataFrame(ALERT).round(3)
res_tbl.to_csv("outputs/forecast_results.csv", index=False); alert_tbl.to_csv("outputs/alert_results.csv", index=False)
print(res_tbl.to_string(index=False)); print(); print(alert_tbl.to_string(index=False))

## 6. The key test: does the model understand cause and effect?

In [ ]:
# ================= WHAT-IF validation: does the model know what a dose change does? =================
# Ground truth is available because we built the simulator: re-run each test patient with the doses on
# days 4 and 5 changed, and compare the TRUE change in INR on day 6 with each model's PREDICTED change.
idx = ids["test"][:600]; DAY = 4
sub = cohort.iloc[idx].reset_index(drop=True); sub_obs = INR_OBS[idx]; sub_dose = DOSE[idx]
p_sub = params_of(sub, U[idx], EPS[idx])
def forecast_all(sched, h=2):
    tw = twin_summary(static_of(sub), sub_obs, sched)
    X = make_rows(sub, sub_obs, sched, tw, [DAY], h); m = MODELS[h]
    return {"Mechanistic twin only": X[f"tw_f{h}"].values, "ML only (no twin)": m["ml"].predict(X[m["bc"]]),
            "Hybrid (twin + ML residual)": X[f"tw_f{h}"].values + m["hyb"].predict(X[m["hc"]])}
base_f = forecast_all(sub_dose)
true_base = simulate(p_sub, OpenLoop(sub_dose))["inr_true"][:, DAY + 2]
wi_pts, per_factor = {}, []
for f in (0.0, 0.5, 0.75, 1.25, 1.5):
    sd = sub_dose.copy(); sd[:, DAY:DAY + 2] = np.round(sd[:, DAY:DAY + 2] * f * 2) / 2
    true_delta = simulate(p_sub, OpenLoop(sd))["inr_true"][:, DAY + 2] - true_base
    new_f = forecast_all(sd)
    for name in base_f:
        wi_pts.setdefault(name, []).append((true_delta, new_f[name] - base_f[name]))
        per_factor.append({"model": name, "dose x": f, "true mean ΔINR": true_delta.mean(), "pred mean ΔINR": (new_f[name] - base_f[name]).mean()})
wi_rows = []
for name, lst in wi_pts.items():
    t = np.concatenate([a for a, _ in lst]); p = np.concatenate([b for _, b in lst]); m = np.abs(t) > 0.05
    wi_rows.append({"model": name, "corr(pred Δ, true Δ)": np.corrcoef(t, p)[0, 1],
                    "direction correct": float(np.mean(np.sign(t[m]) == np.sign(p[m]))), "mean |Δ error| (INR)": float(np.mean(np.abs(t - p)))})
wi_tbl = pd.DataFrame(wi_rows).round(3); wi_tbl.to_csv("outputs/whatif_results.csv", index=False)
pf_tbl = pd.DataFrame(per_factor).round(2)
print(f"What-if test: doses on days {DAY} and {DAY+1} multiplied by 0, 0.5, 0.75, 1.25, 1.5 -> effect on day {DAY+2} INR")
print(wi_tbl.to_string(index=False)); print(); print(pf_tbl.pivot(index="dose x", columns="model", values="pred mean ΔINR").join(
    pf_tbl[pf_tbl["model"] == "Mechanistic twin only"].set_index("dose x")["true mean ΔINR"]).to_string())

In [ ]:
# ================= plots =================
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
r1 = res_tbl[res_tbl["MAE"].notna()]
for h, mk in [("1 day", "o"), ("2 day", "s")]:
    s = r1[r1["horizon"] == h]; ax[0].barh([f"{m} ({h})" for m in s["model"]], s["MAE"])
ax[0].set_xlabel("MAE of forecast INR (lower is better)"); ax[0].set_title("Forecast error by model and horizon")
for name, lst in wi_pts.items():
    t = np.concatenate([a for a, _ in lst]); p = np.concatenate([b for _, b in lst])
    ax[1].scatter(t, p, s=6, alpha=0.35, label=name)
ax[1].plot([-1, 1], [-1, 1], "k--"); ax[1].set_xlim(-1, 1); ax[1].set_ylim(-1, 1)
ax[1].set_xlabel("TRUE change in INR two mornings later"); ax[1].set_ylabel("PREDICTED change in INR"); ax[1].set_title("What-if: dose x0 / 0.5 / 0.75 / 1.25 / 1.5 for 2 nights"); ax[1].legend(fontsize=7)
te1 = DATA[1]["test"]; m1 = MODELS[1]
ph = te1["tw_f1"].values + m1["hyb"].predict(te1[m1["hc"]])
ax[2].scatter(te1["y"], ph, s=3, alpha=0.25); ax[2].plot([0.8, 6], [0.8, 6], "r--"); ax[2].axvline(4, color="gray", ls=":"); ax[2].axhline(4, color="gray", ls=":")
ax[2].set_xlabel("Actual next-morning INR"); ax[2].set_ylabel("Hybrid forecast"); ax[2].set_title("Hybrid, 1-day-ahead (dotted = INR 4)")
plt.tight_layout(); plt.savefig("outputs/forecast_whatif_plots.png", dpi=150); plt.show()

# example patients: observed INR, plus the 1-day-ahead forecast made the morning before (with 90% band)
te1 = te1.copy(); te1["fc"] = ph
fig, axes = plt.subplots(2, 3, figsize=(15, 7)); axes = axes.ravel()
hit = [i for i in ids["test"] if (INR_OBS[i, 1:] >= 4).any()][:3]; ok = [i for i in ids["test"] if not (INR_OBS[i, 1:] >= 4).any()][:3]
for ax_, i in zip(axes, hit + ok):
    ax_.plot(range(N_DAYS), INR_OBS[i], "o-", label="observed INR")
    r = te1[te1["pid"] == i]
    ax_.errorbar(r["day"] + 1, r["fc"], yerr=m1["q90"] * r["fc"], fmt="s", color="orange", capsize=3, label="forecast made day before (90% band)")
    ax_.axhspan(2, 3, color="green", alpha=0.08); ax_.axhline(4, color="red", ls=":")
    ax_.set_title(f"patient {i} | arm: {cohort['arm'][i]} | genotype known: {int(cohort['geno_known'][i])}", fontsize=9)
    ax_.set_xlabel("day"); ax_.set_ylabel("INR")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("outputs/example_patients.png", dpi=150); plt.show()

## 7. The twin as a tool: one patient, several candidate doses

In [ ]:
# ================= the twin as a tool: forecast + dose what-if for ONE patient =================
def forecast_patient(profile, inr_hist, dose_hist, tonight_options, genotype_known=True):
    """Use the TWIN columns to compare doses (it models cause and effect); the HYBRID column is the best forecast for the plan you pick.
       profile: dict(age, height_cm, weight_kg, male, race, cyp2c9, vkorc1, inducer, amiodarone)
       inr_hist: INRs for days 0..d (morning draws) | dose_hist: doses actually given on days 0..d-1 (mg)
       tonight_options: candidate doses (mg) for tonight (day d). Returns next-morning INR forecast + band + P(INR>=4)."""
    d = len(inr_hist) - 1; assert 1 <= d <= 9 and len(dose_hist) == d
    n = len(tonight_options)
    row = pd.DataFrame([{**profile, "age_decade": profile["age"] // 10}])
    hid = row.copy(); hid["cyp2c9"] = "Unknown"; hid["vkorc1"] = "Unknown"
    dtrue, dhid = predict_dose_week(row)[0] / 7.0, predict_dose_week(hid)[0] / 7.0
    df = pd.concat([row] * n, ignore_index=True)
    df["geno_known"] = int(genotype_known); df["dpred_avail"] = dtrue if genotype_known else dhid
    df["cyp_avail"] = profile["cyp2c9"] if genotype_known else "Unknown"; df["vk_avail"] = profile["vkorc1"] if genotype_known else "Unknown"
    inr = np.full((n, N_DAYS), np.nan); inr[:, :d + 1] = np.asarray(inr_hist)[None, :]
    sched = np.zeros((n, N_DAYS - 1)); sched[:, :d] = np.asarray(dose_hist)[None, :]
    for j, opt in enumerate(tonight_options): sched[j, d:] = opt              # assume the option continues
    tw = twin_summary(static_of(df), inr, sched)
    X1 = make_rows(df, inr, sched, tw, [d], 1); X2 = make_rows(df, inr, sched, tw, [d], 2)
    m1_, m2_ = MODELS[1], MODELS[2]
    f1 = X1["tw_f1"].values + m1_["hyb"].predict(X1[m1_["hc"]])             # hybrid: best point forecast for the PLAN
    out = {"tonight_dose_mg": tonight_options,
           "twin INR tomorrow": X1["tw_f1"].values.round(2), "twin INR in 2 days": X2["tw_f2"].values.round(2),
           "twin P(INR>=4 in 2 days)": X2["tw_p4_2"].values.round(3),
           "hybrid INR tomorrow": f1.round(2), "hybrid 90% band": [f"{a:.2f}-{b:.2f}" for a, b in zip(f1 * (1 - m1_["q90"]), f1 * (1 + m1_["q90"]))]}
    return pd.DataFrame(out)

demo_profile = dict(age=72, height_cm=160, weight_kg=62, male=0, race="Asian", cyp2c9="*1/*3", vkorc1="A/A", inducer=0, amiodarone=0)
print("\nDEMO: day-4 morning, INR rising; what happens for different doses tonight (repeated tomorrow night)?")
print(forecast_patient(demo_profile, [1.0, 1.0, 1.1, 1.5, 2.3], [2.0, 2.0, 2.0, 2.0], [0.0, 1.0, 1.5, 2.0, 3.0]).to_string(index=False))

joblib.dump({"models": {h: {k: v for k, v in MODELS[h].items()} for h in MODELS}, "z_known": z_known, "z_hidden": z_hidden,
             "constants": dict(K0=K0, MTT0=MTT0, IMAX=IMAX, GAMMA=GAMMA, TARGET_INR=TARGET_INR, CL_MULT=CL_MULT, VAR_SCALE=VAR_SCALE)},
            "models/inr_forecaster_bundle.joblib")